In [0]:
%pip install boto3 -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, boto3
os.environ["AWS_ACCESS_KEY_ID"] = "Your Access Key "
os.environ["AWS_SECRET_ACCESS_KEY"] = "Your Secret Access Key "
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
s3 = boto3.client("s3")
BUCKET = "nbapredictions-sthomas26-ncf"
SEASON = "2024-25"

In [0]:
import os
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NBA-Gold").getOrCreate()

# Download silver parquet files using boto3
os.makedirs("/tmp/silver", exist_ok=True)
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"silver/games/season={SEASON}/")
for obj in resp["Contents"]:
    key = obj["Key"]
    fname = key.split("/")[-1]
    if fname.endswith(".parquet"):
        s3.download_file(BUCKET, key, f"/tmp/silver/{fname}")

# Load with pandas (serverless doesn't allow Spark to read /tmp/)
pandas_df = pd.read_parquet("/tmp/silver/")
silver_df = spark.createDataFrame(pandas_df)
print(f"Silver rows loaded: {silver_df.count()}")

Silver rows loaded: 2802


In [0]:
from pyspark.sql.functions import lag, avg, when, datediff, col
from pyspark.sql.window import Window

w = Window.partitionBy("TEAM_ID").orderBy("GAME_DATE")

gold_df = silver_df \
    .withColumn("HOME", when(col("MATCHUP").contains("vs."), 1).otherwise(0)) \
    .withColumn("WIN", when(col("WL") == "W", 1).otherwise(0)) \
    .withColumn("PREV_GAME_DATE", lag("GAME_DATE", 1).over(w)) \
    .withColumn("REST_DAYS", datediff(col("GAME_DATE"), col("PREV_GAME_DATE"))) \
    .withColumn("AVG_PTS_L10", avg("PTS").over(w.rowsBetween(-10, -1))) \
    .withColumn("AVG_REB_L10", avg("REB").over(w.rowsBetween(-10, -1))) \
    .withColumn("AVG_AST_L10", avg("AST").over(w.rowsBetween(-10, -1))) \
    .withColumn("AVG_TOV_L10", avg("TOV").over(w.rowsBetween(-10, -1))) \
    .withColumn("AVG_PLUS_MINUS_L10", avg("PLUS_MINUS").over(w.rowsBetween(-10, -1))) \
    .dropna()

print(f"Gold rows: {gold_df.count()}")
gold_df.show(5)

Gold rows: 2757
+----------+----------+----------+-----------------+--------------------+-----------+---+---+------+-------+------+---+---+---+----------+----+---+--------------+---------+------------------+------------------+-----------+------------------+------------------+
|   GAME_ID| GAME_DATE|   TEAM_ID|TEAM_ABBREVIATION|           TEAM_NAME|    MATCHUP| WL|PTS|FG_PCT|FG3_PCT|FT_PCT|REB|AST|TOV|PLUS_MINUS|HOME|WIN|PREV_GAME_DATE|REST_DAYS|       AVG_PTS_L10|       AVG_REB_L10|AVG_AST_L10|       AVG_TOV_L10|AVG_PLUS_MINUS_L10|
+----------+----------+----------+-----------------+--------------------+-----------+---+---+------+-------+------+---+---+---+----------+----+---+--------------+---------+------------------+------------------+-----------+------------------+------------------+
|0012400011|2024-10-07|     15020|              NZB|New Zealand Breakers|  NZB @ PHI|  L| 84| 0.361|  0.143|   0.8| 31| 11| 20|     -55.0|   0|  0|    2024-10-04|        3|              87.0|          

In [0]:
import tempfile
import pandas as pd

# Convert to pandas and create a clean copy to remove Spark metadata
pandas_df = pd.DataFrame(gold_df.toPandas())

# Use Python's tempfile (not Spark's /tmp) to avoid DBFS restriction
with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = f"{tmpdir}/gold_games.parquet"
    pandas_df.to_parquet(parquet_path, index=False)
    
    # Upload to S3
    s3.upload_file(
        parquet_path,
        BUCKET,
        f"gold/games/season={SEASON}/games.parquet"
    )
    print(f"uploaded → gold/games/season={SEASON}/games.parquet")

uploaded → gold/games/season=2024-25/games.parquet
